In [1]:
import os
from pypdf import PdfReader

In [2]:
all_pdf_data = [] # A list to hold text from ALL files
for filename in os.listdir("data"):
    # 2. Check if the file is a PDF (ignoring if it's .pdf or .PDF)
    if filename.lower().endswith(".pdf"):
        print(f"Processing: {filename}")

        file_path = os.path.join("data", filename)

        reader = PdfReader(file_path)
        file_text = ""
        for page in reader.pages:
            file_text += page.extract_text() + " "

        all_pdf_data.append({
            "source": filename,
            "content": file_text
        })

        first_page_text = reader.pages[0].extract_text()
        print(f"--- Document: {filename} ---")
        print(f"Total Pages: {len(reader.pages)}")
        print(f"Preview: {first_page_text[:200]}...")
        print("-" * 30)

print(f"\nSuccess! Processed {len(all_pdf_data)} documents.")

Processing: elise.pdf
--- Document: elise.pdf ---
Total Pages: 12
Preview: Can you walk me through your background? 
I’m Praj. I recently graduated from UC Berkeley with a master’s degree in information 
systems. I have a background in customer-facing analytics and consultin...
------------------------------
Processing: roku.pdf
--- Document: roku.pdf ---
Total Pages: 23
Preview: Can you walk me through your background? 
I’m Praj. I recently graduated from UC Berkeley with a master’s degree in Information 
Systems. 
I interned at Roku in 2024, working closely with UX, product,...
------------------------------

Success! Processed 2 documents.


In [22]:
import re

def smart_chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    
    while start < len(text):
        # 1. Determine the 'ideal' end
        end = start + chunk_size
        
        # 2. If not at the very end, find the last space to avoid cutting a word
        if end < len(text):
            search_area = text[end-100:end]
            last_space = max(search_area.rfind(" "), search_area.rfind("\n"))
            if last_space != -1:
                end = (end - 100) + last_space
        
        # 3. Extract the chunk
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        
        # 4. CRITICAL FIX: Calculate the next start point
        # Jump back by overlap, then find the NEXT space so we don't start mid-word
        next_start = end - overlap
        if next_start < 0:
            next_start = 0
            
        # Look forward from the overlap point to find the first space
        # This ensures the NEW chunk starts at the beginning of a word
        first_space = text.find(" ", next_start)
        if first_space != -1 and first_space < end:
            start = first_space + 1
        else:
            start = end # Fallback if no space is found
            
    return chunks

In [23]:
all_chunks = []

for doc in all_pdf_data: # Loop through the PDFs we processed earlier
    filename = doc["source"] #to be used like a key
    content = doc["content"] #to be chunked
    
    # Chunk this specific PDF's text
    chunks = smart_chunk_text(content)
    
    for chunk in chunks:
        # Save each chunk with its source name
        all_chunks.append({
            "source": filename,
            "text": chunk
        })

print(f"Total chunks created from all PDFs: {len(all_chunks)}")
print(f"Example chunk from {all_chunks[0]['source']}:")
print(all_chunks[0]['text'][:100])

Total chunks created from all PDFs: 135
Example chunk from elise.pdf:
Can you walk me through your background? 
I’m Praj. I recently graduated from UC Berkeley with a mas


In [ ]:
import chromadb
from chromadb.utils import embedding_functions

# Initializing vector Database (creates a 'db' folder in project)
client = chromadb.PersistentClient(path="./chroma_db")

# Defining the Embedding Function (So Chroma knows how to turn text into numbers)
# using a small model from hugging face for fast embedding
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# Creating a 'Collection' (like Table)
# get_or_create prevents errors if run this cell twice
collection = client.get_or_create_collection(name="pdf_knowledge_base", embedding_function=sentence_transformer_ef)



Vector Database is ready! Added 136 chunks.


In [24]:

# Refresh all existing data in this specific collection
try:
    client.delete_collection(name="pdf_knowledge_base")
    print("Old collection deleted.")
except:
    print("No old collection found. Creating fresh.")

collection = client.get_or_create_collection(
    name="pdf_knowledge_base", 
    embedding_function=sentence_transformer_ef
)

# Preparing data for the database
# genereating unique IDs for every chunk
ids = [f"id_{i}" for i in range(len(all_chunks))]
documents = [chunk["text"] for chunk in all_chunks]
metadatas = [{"source": chunk["source"]} for chunk in all_chunks]

#Adding everything to the Vector DB
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"Vector Database is ready! Added {collection.count()} chunks.")

Old collection deleted.
Vector Database is ready! Added 135 chunks.


In [27]:
results = collection.query(
    query_texts=["What is the your experience with ambiguity?"], # Ask your question here
    n_results=3 # This is our 'Top K' - it returns the 3 best matches
)

# Print the results and their sources
for i in range(len(results['documents'][0])):
    print(f"Match {i+1} (Source: {results['metadatas'][0][i]['source']}):")
    print(f"{results['documents'][0][i][:500]}...\n")

Match 1 (Source: elise.pdf):
How do you work cross-functionally when there’s tension? 
S: During my time with ZS, one of our customers was going through large-scale 
organizational restructuring affecting sales teams 
T: There was significant ambiguity regarding the timeline and exact data impacts 
A: I took the initiative to set up a cross-functional alignment meeting with upstream and 
downstream teams. to align on the timeline, constraints, and ownership of changes. 
[Emphasizing shared goals of smooth customer...

Match 2 (Source: roku.pdf):
a time you managed conflicting stakeholders. 
I start by understanding the underlying goal behind the request. Often the disagreement 
isn’t about the metric or feature, but the outcome they care about.  S: During the Omnichannel project at ZS, my stakeholders had conflicting opinions on how 
to define the metric Engagements per week.  
T: I needed to define the best approach for our users, so that we could kick off 
development. 
A: I facilita